Basic PyACP Example {#basic_flat_plate}
===================

Define a Composite Lay-up with PyACP, solve the resulting model with
PyMAPDL, and run a failure analysis with PyDPF-Composites.

The starting point is a MAPDL \*.dat file which contains the mesh,
material data and the boundary conditions. The
`input_file_for_pyacp`{.interpreted-text role="ref"} section describes
how input files can be created. The \*.dat file is imported in PyACP to
define the lay-up. PyACP exports the resulting model for PyMAPDL. Once
the results are available, the RST file is loaded in PyDPF composites.
The additional input files (material.xml and ACPCompositeDefinitions.h5)
can also be stored with PyACP and passed to PyDPF Composites.


Import standard library and third-party dependencies.


In [ ]:
import pathlib
import tempfile

Import PyACP dependencies


In [ ]:
from ansys.acp.core import (
    ACPWorkflow,
    PlyType,
    example_helpers,
    get_composite_post_processing_files,
    get_directions_plotter,
    get_dpf_unit_system,
    launch_acp,
    material_property_sets,
    print_model,
)

Get example file from server


In [ ]:
tempdir = tempfile.TemporaryDirectory()
WORKING_DIR = pathlib.Path(tempdir.name)
input_file = example_helpers.get_example_file(
    example_helpers.ExampleKeys.BASIC_FLAT_PLATE_DAT, WORKING_DIR
)

Launch the PyACP server and connect to it.


In [ ]:
acp = launch_acp()

Define the input file and instantiate an ACPWorkflow. The ACPWorkflow
class provides convenience methods which simplify the file handling. It
automatically creates a model based on the input file.


In [ ]:
workflow = ACPWorkflow.from_cdb_or_dat_file(
    acp=acp,
    cdb_or_dat_file_path=input_file,
    local_working_directory=WORKING_DIR,
)

model = workflow.model
print(workflow.working_directory.path)
print(model.unit_system)

Visualize the loaded mesh.


In [ ]:
mesh = model.mesh.to_pyvista()
mesh.plot(show_edges=True)

Create an orthotropic material and fabric including strain limits, which
are later used to post-process the simulation.


In [ ]:
engineering_constants = (
    material_property_sets.ConstantEngineeringConstants.from_orthotropic_constants(
        E1=5e10, E2=1e10, E3=1e10, nu12=0.28, nu13=0.28, nu23=0.3, G12=5e9, G23=4e9, G31=4e9
    )
)

strain_limit = 0.01
strain_limits = material_property_sets.ConstantStrainLimits.from_orthotropic_constants(
    eXc=-strain_limit,
    eYc=-strain_limit,
    eZc=-strain_limit,
    eXt=strain_limit,
    eYt=strain_limit,
    eZt=strain_limit,
    eSxy=strain_limit,
    eSyz=strain_limit,
    eSxz=strain_limit,
)

ud_material = model.create_material(
    name="UD",
    ply_type=PlyType.REGULAR,
    engineering_constants=engineering_constants,
    strain_limits=strain_limits,
)

fabric = model.create_fabric(name="UD", material=ud_material, thickness=0.1)

Define a rosette and an oriented selection set and plot the
orientations.


In [ ]:
rosette = model.create_rosette(origin=(0.0, 0.0, 0.0), dir1=(1.0, 0.0, 0.0), dir2=(0.0, 0.0, 1.0))

oss = model.create_oriented_selection_set(
    name="oss",
    orientation_point=(0.0, 0.0, 0.0),
    orientation_direction=(0.0, 1.0, 0),
    element_sets=[model.element_sets["All_Elements"]],
    rosettes=[rosette],
)

model.update()

orientation = oss.elemental_data.orientation
assert orientation is not None
plotter = get_directions_plotter(model=model, components=[orientation])
plotter.show()

Create various plies with different angles and add them to a modeling
group.


In [ ]:
modeling_group = model.create_modeling_group(name="modeling_group")
angles = [0, 45, -45, 45, -45, 0]
for idx, angle in enumerate(angles):
    modeling_group.create_modeling_ply(
        name=f"ply_{idx}_{angle}_{fabric.name}",
        ply_angle=angle,
        ply_material=fabric,
        oriented_selection_sets=[oss],
    )

model.update()

Show the fiber directions of a specific ply.


In [ ]:
modeling_ply = model.modeling_groups["modeling_group"].modeling_plies["ply_4_-45_UD"]


fiber_direction = modeling_ply.elemental_data.fiber_direction
assert fiber_direction is not None
plotter = get_directions_plotter(
    model=model,
    components=[fiber_direction],
)

plotter.show()

Print the model tree for a quick overview. Note: The model can also be
opened in the ACP GUI. See
`view_the_model_in_the_acp_gui`{.interpreted-text role="ref"}.


In [ ]:
print_model(model)

Solve the model with MAPDL
==========================

Launch the MAPDL instance.


In [ ]:
from ansys.mapdl.core import launch_mapdl

mapdl = launch_mapdl()
mapdl.clear()

Load the CDB file into PyMAPDL.


In [ ]:
mapdl.input(str(workflow.get_local_cdb_file()))

Solve the model


In [ ]:
mapdl.allsel()
mapdl.slashsolu()
mapdl.solve()

Post-processing: show displacements.


In [ ]:
mapdl.post1()
mapdl.set("last")
mapdl.post_processing.plot_nodal_displacement(component="NORM")

Download the rst file for composite specific post-processing.


In [ ]:
rstfile_name = f"{mapdl.jobname}.rst"
rst_file_local_path = workflow.working_directory.path / rstfile_name
mapdl.download(rstfile_name, str(workflow.working_directory.path))

Post-Processing with DPF composites
===================================

Setup: configure imports and connect to the pyDPF Composites server and
load the dpf composites plugin.


In [ ]:
from ansys.dpf.composites.composite_model import CompositeModel
from ansys.dpf.composites.constants import FailureOutput
from ansys.dpf.composites.failure_criteria import CombinedFailureCriterion, MaxStrainCriterion
from ansys.dpf.composites.server_helpers import connect_to_or_start_server

Connect to the server. The `connect_to_or_start_server` function
automatically loads the composites plugin.


In [ ]:
dpf_server = connect_to_or_start_server()

Specify the Combined Failure Criterion.


In [ ]:
max_strain = MaxStrainCriterion()

cfc = CombinedFailureCriterion(
    name="Combined Failure Criterion",
    failure_criteria=[max_strain],
)

Create the CompositeModel and configure its input.


In [ ]:
composite_model = CompositeModel(
    get_composite_post_processing_files(workflow, rst_file_local_path),
    default_unit_system=get_dpf_unit_system(model.unit_system),
    server=dpf_server,
)

Evaluate the failure criteria and plot it.


In [ ]:
output_all_elements = composite_model.evaluate_failure_criteria(cfc)
irf_field = output_all_elements.get_field({"failure_label": FailureOutput.FAILURE_VALUE})
irf_field.plot()

Release composite model to close open streams to result file.


In [ ]:
composite_model = None  # type: ignore